# Assignment 2: Policy Q&A Bot

This notebook implements a retrieval-first policy Q&A system using policy files from Google Drive (with local fallback).

Run cells in order from top to bottom.

In [ ]:
# Cell 1: Imports and Environment Check
import re
import math
import json
import platform
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Set
from urllib.parse import parse_qs, urlparse
from urllib.request import urlopen

import pandas as pd

print("Python:", platform.python_version())
print("Platform:", platform.platform())

Python: 3.12.10
Platform: Windows-11-10.0.26200-SP0


In [ ]:
# Cell 2: Input/Config Variables
BASE_DIR = Path.cwd()
POLICY_FILES = {
    'leave_policy.txt': 'Leave Policy',
    'it_policy.txt': 'IT Policy',
    'travel_policy.txt': 'Travel Policy',
}

# Google Drive mode: yahan har file ka Drive share URL dalo (public/viewable link).
DRIVE_FILES = {
    'leave_policy.txt': 'https://drive.google.com/file/d/17V-2ryfPQQ-KKhlLF_SCPMKVU2v8Kqyg/view?usp=sharing',
    'it_policy.txt': 'https://drive.google.com/file/d/1FBWWZgAfYkBo5Z_I4RHSRraVnzaNh7XT/view?usp=sharing',
    'travel_policy.txt': 'https://drive.google.com/file/d/1FqUPC2ExAw9dUOYTWzPRtP3SFL51y953/view?usp=sharing',
}
USE_DRIVE = True

FALLBACK_MESSAGE = 'Information not available in policy documents.'
SEMANTIC_WEIGHT = 0.7
TFIDF_WEIGHT = 0.3
BASE_SCORE_THRESHOLD = 0.12

PROCESS_QUERY_PATTERN = re.compile(r"\b(how\s+(to|can|do)|step[-\s]?by[-\s]?step|process|procedure|kaise)\b", re.IGNORECASE)
QUANT_QUERY_PATTERN = re.compile(r"\bhow\s+many\b", re.IGNORECASE)
PROCESS_DETAIL_PATTERN = re.compile(r"\b(step|submit|apply|form|portal|workflow|email|request|approval chain|approver)\b", re.IGNORECASE)

QUERY_EXPANSIONS = {
    'manager': ['managerial', 'approval authority'],
    'managerial': ['manager', 'approval authority'],
    'approval': ['approve', 'authorization', 'sign-off'],
    'approve': ['approval', 'authorization', 'sign-off'],
    'remote': ['work from home', 'wfh'],
    'reimbursement': ['claim', 'refund'],
    'international': ['abroad', 'overseas'],
    'abroad': ['international', 'overseas'],
    'overseas': ['international', 'abroad'],
    'travel': ['trip', 'journey'],
    'leave': ['time off', 'vacation'],
}

STOPWORDS = {
    'a','an','and','are','as','at','be','by','for','from','how','i','in','is','it','me','my','of','on','or',
    'that','the','to','was','what','when','where','who','why','with','do','does','can','should','necessary','need','required','going'
}

RESULT_JSON = BASE_DIR / 'assignment2_results.json'
RESULT_CSV = BASE_DIR / 'assignment2_results.csv'


def extract_drive_file_id(url: str) -> str:
    parsed = urlparse(url)

    if parsed.netloc in {'drive.google.com', 'www.drive.google.com'}:
        parts = parsed.path.split('/')
        if 'd' in parts:
            d_idx = parts.index('d')
            if d_idx + 1 < len(parts):
                return parts[d_idx + 1]

        query_id = parse_qs(parsed.query).get('id', [])
        if query_id:
            return query_id[0]

    if parsed.netloc in {'docs.google.com', 'www.docs.google.com'}:
        query_id = parse_qs(parsed.query).get('id', [])
        if query_id:
            return query_id[0]

    return ''


def drive_direct_download_url(share_url: str) -> str:
    file_id = extract_drive_file_id(share_url.strip())
    if not file_id:
        return ''
    return f'https://drive.google.com/uc?export=download&id={file_id}'


def read_text_from_drive(share_url: str) -> str:
    direct_url = drive_direct_download_url(share_url)
    if not direct_url:
        raise ValueError('Invalid Google Drive URL. Expected /file/d/<id>/view or ?id=<id>.')

    with urlopen(direct_url, timeout=30) as response:
        raw_bytes = response.read()

    return raw_bytes.decode('utf-8', errors='ignore')


def get_policy_text(file_name: str, base_dir: Path) -> str:
    drive_url = DRIVE_FILES.get(file_name, '').strip()

    if USE_DRIVE and drive_url:
        try:
            return read_text_from_drive(drive_url)
        except Exception as exc:
            print(f'Drive read failed for {file_name}: {exc}')
            print('Trying local fallback...')

    local_path = base_dir / file_name
    if local_path.exists():
        return local_path.read_text(encoding='utf-8', errors='ignore')

    raise FileNotFoundError(f'{file_name} not found in Drive config or local path: {local_path}')


def ensure_policy_sources_available(base_dir: Path, file_names: list[str]) -> None:
    missing_sources = []

    for name in file_names:
        has_drive = bool(DRIVE_FILES.get(name, '').strip())
        has_local = (base_dir / name).exists()

        if USE_DRIVE and has_drive:
            continue
        if has_local:
            continue

        missing_sources.append(name)

    if not missing_sources:
        source_mode = 'Google Drive + local fallback' if USE_DRIVE else 'Local files'
        print('Policy source check passed. Mode:', source_mode)
        return

    print('Missing sources for files:', ', '.join(missing_sources))
    print('Either set DRIVE_FILES URLs, or place files in:', base_dir)


print('Config initialized. Working dir:', BASE_DIR.resolve())
print('USE_DRIVE =', USE_DRIVE)
print('Set DRIVE_FILES URLs in this cell before running Cell 4.')

Config initialized. Base dir: C:\Users\MY HP\Downloads\Assesment-v1\Assesment-v1


In [ ]:
# Cell 3: Core Functions
@dataclass
class PolicyChunk:
    document_file: str
    document_name: str
    text: str


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def build_tfidf_vector(tokens: List[str], idf_map: Dict[str, float]) -> Dict[str, float]:
    if not tokens:
        return {}
    term_counts = Counter(tokens)
    total_terms = len(tokens)
    vector: Dict[str, float] = {}
    for token, count in term_counts.items():
        if token in idf_map:
            vector[token] = (count / total_terms) * idf_map[token]
    return vector


def cosine_sim_sparse(vec_a: Dict[str, float], vec_b: Dict[str, float]) -> float:
    if not vec_a or not vec_b:
        return 0.0
    dot = sum(value * vec_b.get(key, 0.0) for key, value in vec_a.items())
    norm_a = math.sqrt(sum(v * v for v in vec_a.values()))
    norm_b = math.sqrt(sum(v * v for v in vec_b.values()))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)


def semantic_token_set(question: str) -> Set[str]:
    lowered = question.lower()
    tokens = set(tokenize(question))
    expanded = {t for t in tokens if t not in STOPWORDS and len(t) > 2}

    for term, replacements in QUERY_EXPANSIONS.items():
        replacement_tokens = set()
        for replacement in replacements:
            replacement_tokens.update(tokenize(replacement))

        if (
            term in lowered
            or term in tokens
            or any(rep in lowered for rep in replacements)
            or any(rt in tokens for rt in replacement_tokens)
        ):
            expanded.add(term)

    return expanded


def semantic_overlap_score(query_tokens: Set[str], chunk_tokens: Set[str]) -> float:
    if not query_tokens or not chunk_tokens:
        return 0.0

    chunk_filtered = {token for token in chunk_tokens if token not in STOPWORDS and len(token) > 2}
    if not chunk_filtered:
        return 0.0

    intersection = len(query_tokens.intersection(chunk_filtered))
    coverage = intersection / max(1, len(query_tokens))
    precision = intersection / max(1, len(chunk_filtered))
    return min(1.0, (0.75 * coverage) + (0.25 * precision))


def load_policy_chunks(base_dir: Path) -> List[PolicyChunk]:
    chunks: List[PolicyChunk] = []
    for file_name, doc_name in POLICY_FILES.items():
        try:
            raw_text = get_policy_text(file_name, base_dir)
        except Exception as exc:
            print(f"Skipping {file_name}: {exc}")
            continue

        lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
        for line in lines:
            if line.lower().endswith('policy'):
                continue
            cleaned = re.sub(r'^\d+\.\s*', '', line).strip()
            if cleaned:
                chunks.append(PolicyChunk(file_name, doc_name, cleaned))
    return chunks


def build_retriever(chunks: List[PolicyChunk]):
    tokenized_chunks: List[List[str]] = [tokenize(chunk.text) for chunk in chunks]
    doc_freq: Counter = Counter()
    for tokens in tokenized_chunks:
        doc_freq.update(set(tokens))

    total_docs = len(tokenized_chunks)
    idf_map: Dict[str, float] = {
        token: math.log((1 + total_docs) / (1 + frequency)) + 1.0
        for token, frequency in doc_freq.items()
    }

    chunk_tfidf_vectors = [build_tfidf_vector(tokens, idf_map) for tokens in tokenized_chunks]
    chunk_token_sets = [set(tokens) for tokens in tokenized_chunks]
    return idf_map, chunk_tfidf_vectors, chunk_token_sets


def build_query_variants(question: str) -> List[str]:
    lowered = question.lower().strip()
    variants = {lowered}
    for term, replacements in QUERY_EXPANSIONS.items():
        if term in lowered:
            for replacement in replacements:
                variants.add(lowered.replace(term, replacement))
    return list(variants)


def is_answer_adequate(question: str, answer_text: str, score: float) -> bool:
    if score < BASE_SCORE_THRESHOLD:
        return False

    is_quant_query = QUANT_QUERY_PATTERN.search(question) is not None
    is_process_query = PROCESS_QUERY_PATTERN.search(question) is not None

    if is_process_query and not is_quant_query:
        return PROCESS_DETAIL_PATTERN.search(answer_text) is not None
    return True


def find_best_match(question: str, chunks: List[PolicyChunk]):
    if not chunks:
        return None

    idf_map, chunk_tfidf_vectors, chunk_token_sets = build_retriever(chunks)
    query_variants = build_query_variants(question)

    tfidf_scores = [0.0] * len(chunks)
    semantic_scores = [0.0] * len(chunks)

    for variant in query_variants:
        query_tokens = tokenize(variant)
        query_tfidf_vector = build_tfidf_vector(query_tokens, idf_map)
        expanded_tokens = semantic_token_set(variant)

        for idx in range(len(chunks)):
            tfidf_score = cosine_sim_sparse(query_tfidf_vector, chunk_tfidf_vectors[idx])
            semantic_score = semantic_overlap_score(expanded_tokens, chunk_token_sets[idx])
            tfidf_scores[idx] = max(tfidf_scores[idx], tfidf_score)
            semantic_scores[idx] = max(semantic_scores[idx], semantic_score)

    blended_scores = [
        (SEMANTIC_WEIGHT * semantic_scores[idx]) + (TFIDF_WEIGHT * tfidf_scores[idx])
        for idx in range(len(chunks))
    ]

    best_idx = max(range(len(chunks)), key=lambda i: blended_scores[i])
    best_score = float(blended_scores[best_idx])
    best_chunk = chunks[best_idx]

    if not is_answer_adequate(question, best_chunk.text, best_score):
        return None

    return {
        'question': question,
        'answer': best_chunk.text,
        'document': best_chunk.document_name,
        'document_file': best_chunk.document_file,
        'score': best_score,
        'semantic_score': float(semantic_scores[best_idx]),
        'tfidf_score': float(tfidf_scores[best_idx]),
        'retrieval_mode': 'hybrid',
    }

print('Core functions ready.')

Core functions ready.


In [ ]:
# Cell 4: Execute Main Processing
ensure_policy_sources_available(BASE_DIR, list(POLICY_FILES.keys()))

chunks = load_policy_chunks(BASE_DIR)
print('Loaded chunks:', len(chunks))
if len(chunks) == 0:
    print('No chunks loaded. Set valid DRIVE_FILES links (if USE_DRIVE=True) or keep local files and rerun this cell.')

test_queries = [
    'Is VPN mandatory for remote work?',
    'How many sick leaves can employee take?',
    'What is hotel reimbursement cap?',
    'How to apply for approval step-by-step?',
    'What is dress code policy?'
]

results = []
for query in test_queries:
    match = find_best_match(query, chunks)
    if match is None:
        results.append({
            'question': query,
            'answer': FALLBACK_MESSAGE,
            'document': 'N/A',
            'document_file': 'N/A',
            'score': 0.0,
            'semantic_score': 0.0,
            'tfidf_score': 0.0,
            'retrieval_mode': 'fallback',
        })
    else:
        results.append(match)

print('Processed queries:', len(results))

Loaded chunks: 15
Processed queries: 5


In [5]:
# Cell 5: Display Results
results_df = pd.DataFrame(results)
results_df

,question,answer,document,document_file,score,semantic_score,tfidf_score,retrieval_mode
0,Is VPN mandatory for remote work?,VPN is mandatory for remote work access.,IT Policy,it_policy.txt,0.938148,0.950000,0.910494,hybrid
1,How many sick leaves can employee take?,Sick leave entitlement is 10 days per year.,Leave Policy,leave_policy.txt,0.319585,0.333333,0.287506,hybrid
2,What is hotel reimbursement cap?,Hotel reimbursement limit is Rs. 5000 per night.,Travel Policy,travel_policy.txt,0.586193,0.583333,0.592865,hybrid
3,How to apply for approval step-by-step?,Information not available in policy documents.,N/A,N/A,0.000000,0.000000,0.000000,fallback
4,What is dress code policy?,Information not available in policy documents.,N/A,N/A,0.000000,0.000000,0.000000,fallback


In [6]:
# Cell 6: Save Output to File
results_df.to_csv(RESULT_CSV, index=False)
with RESULT_JSON.open('w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print('Saved:', RESULT_CSV.resolve())
print('Saved:', RESULT_JSON.resolve())

Saved: C:\Users\MY HP\Downloads\Assesment-v1\Assesment-v1\assignment2_results.csv
Saved: C:\Users\MY HP\Downloads\Assesment-v1\Assesment-v1\assignment2_results.json


In [ ]:
# Cell 7: Quick Validations (Assertions)
if len(chunks) == 0:
    print('Validation skipped: No policy chunks loaded yet.')
    print('Set valid DRIVE_FILES links (or keep local files), then rerun Cell 4.')
else:
    assert len(results) == len(test_queries), 'Results count mismatch.'
    assert results_df.shape[0] == len(test_queries), 'DataFrame row count mismatch.'
    assert 'answer' in results_df.columns, 'Missing answer column.'
    assert (results_df['answer'].str.len() > 0).all(), 'Empty answer found.'
    print('All quick validations passed.')

All quick validations passed.


## Final Interactive Input (Same Retrieval Logic)
Run this cell and type your custom question to get answer, source document, and hybrid scores.
If answer is not found, exact fallback message is returned.

In [ ]:
# Cell 10: Optional Sample Queries (Reference)
sample_queries = [
    "Is VPN mandatory for remote work?",
    "How many sick leaves can employee take?",
    "What is hotel reimbursement cap?",
    "How to apply for approval step-by-step?",
    "What is dress code policy?"
]
print("Sample queries:")
for item in sample_queries:
    print("-", item)

Question set: Is VPN mandatory for remote work?


In [ ]:
# Cell 11 (Last Cell): Interactive User Input + Output
try:
    user_question = input("Enter your question: ").strip()
except EOFError:
    user_question = ""

if not user_question:
    print("Please enter a question.")
else:
    match = find_best_match(user_question, chunks)
    if not match:
        print(FALLBACK_MESSAGE)
    else:
        print(match['answer'])
        print(f"Source document: {match['document']} ({match['document_file']})")
        print(
            f"Hybrid score: {match['score']:.3f} | "
            f"Semantic: {match['semantic_score']:.3f} | "
            f"TF-IDF: {match['tfidf_score']:.3f}"
        )